# AI Lab - Practical Question No. 5
## Utility-Based Agent Simulation: Path Cost Optimization

- **Syllabus Area**: Intelligent Agents
- **Difficulty**: 🟠 Advanced
- **Practical Type**: Optimization
- **Core AI Concept**: Utility-Based Agent, Utility Functions, Cost Minimization, PEAS Framework
- **Python Concepts Tested**: Dictionary iteration, custom sorting lambda functions
- **Complexity**: Time Complexity: $O(P \times L)$ | Space Complexity: $O(1)$ where $P$ is number of paths and $L$ is path length

---

### Problem Statement
Write a Python program to simulate a **Utility-Based Agent** that must choose a path to a goal.
There are multiple valid paths, but each path has different terrain costs (e.g., Grass=1, Mud=3, Water=5).
The agent must use a utility function to evaluate and select the path that maximizes utility (minimizes total cost).

**Task Requirements**:
1. Define a graph representing nodes and terrain edge costs.
2. Formulate a utility function $U(path) = -1 \times total\_cost$.
3. Evaluate 3 predefined paths.
4. Output the path with the highest utility.

**Sample Input**: `Paths: [A-B-D (cost 6), A-C-D (cost 4)]`  
**Sample Output**: `Selected Path: A-C-D, Utility: -4`

**Optional Challenge**: Add battery constraint—reject paths exceeding a certain cost regardless of utility.

### Agent PEAS Specification

| PEAS Component | Specification for Question 5 |
| :--- | :--- |
| **P**erformance Measure | Maximize utility $U(p) = -1 \times \text{cost}(p)$; minimize energy expenditure; respect battery budget |
| **E**nvironment | Weighted directed graph with terrain edge costs (Grass=1, Mud=3, Water=5) |
| **A**ctuators | Path selection and waypoint traversal commands |
| **S**ensors | Graph topography dictionary, terrain identification sensors, battery gauge |

### Mathematical Formulation: Cost Minimization vs. Utility Maximization

In classical decision theory and intelligent agents (Russell & Norvig, Chapter 2 & 16), a utility function $U: S \to \mathbb{R}$ quantifies the agent's degree of preference.

When traversing terrain, each edge transition $e = (u, v)$ incurs an energy cost $c(u, v)$:
$$\text{total\_cost}(p) = \sum_{i=1}^{L-1} c(v_i, v_{i+1})$$

Because costs represent loss/expenditure, the agent formulates utility by negating total cost:
$$U(p) = -1 \times \text{total\_cost}(p)$$

This linear inversion ensures that:
$$\arg\max_{p} U(p) \iff \arg\min_{p} \text{total\_cost}(p)$$

#### Terrain Cost Schedule
| Terrain Type | Traversal Cost | Physical Characteristics |
| :--- | :--- | :--- |
| **Grass** | `1.0` | Firm, dry ground with minimal rolling resistance |
| **Sand** | `2.0` | Loose particulate ground causing moderate slip |
| **Mud** | `3.0` | High-viscosity soil with significant resistance |
| **Rock** | `4.0` | Uneven boulders requiring complex obstacle avoidance |
| **Water** | `5.0` | Deep water crossing with extreme drag and energy drain |

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on sys.path when running from notebooks/
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utility_based_agent import (
    DEFAULT_GRAPH,
    DEFAULT_PREDEFINED_PATHS,
    DEFAULT_TERRAIN_COSTS,
    PathEvaluation,
    UtilityBasedAgent,
    UtilitySimulationResult,
    calculate_path_cost,
    calculate_utility,
    find_all_paths,
    resolve_edge_cost,
    run_utility_based_simulation,
)
print("Utility-Based Agent modules imported successfully.")

## 1. Graph Topography & Edge Cost Definition

Representing nodes and terrain edge costs as a nested dictionary structure `graph[u][v]`:

In [ ]:
print("Graph Adjacency Topography:")
for u, neighbors in DEFAULT_GRAPH.items():
    print(f"  Node [{u}]:")
    for v, edge in neighbors.items():
        cost = edge.get('cost') if isinstance(edge, dict) else edge
        terrain = edge.get('terrain', 'Direct') if isinstance(edge, dict) else 'Direct'
        print(f"    -> [{v}] Terrain: {terrain:<6} | Cost: {cost}")

## 2. Evaluation of 3 Predefined Paths (Task Requirements 2, 3 & 4)

Evaluating the 3 predefined paths connecting Start `A` to Goal `D`:
1. `A-B-D`: Grass ($1$) + Water ($5$) = Total Cost $6$ $\to$ Utility $-6$
2. `A-C-D`: Grass ($1$) + Mud ($3$) = Total Cost $4$ $\to$ Utility $-4$ **(Highest Utility / Optimal)**
3. `A-E-D`: Mud ($3$) + Water ($5$) = Total Cost $8$ $\to$ Utility $-8$

In [ ]:
result = run_utility_based_simulation()

print("Candidate Path Evaluations:")
print("-" * 70)
print(f"{'#':<3} {'Path':<14} {'Cost':<8} {'Utility':<10} {'Status'}")
print("-" * 70)
for i, ev in enumerate(result.evaluated_paths, start=1):
    status = "SELECTED (Optimal)" if ev.path == result.selected_path.path else "FEASIBLE"
    print(f"{i:<3} {ev.path_str:<14} {ev.total_cost:<8.1f} {ev.utility:<10.1f} {status}")
print("-" * 70)

print(f"\nSample Output: Selected Path: {result.selected_path.path_str}, Utility: {result.selected_path.utility:.0f}")
print(f"Total Cost:    {result.selected_path.total_cost}")

## 3. Custom Sorting & Optimization via Lambda Functions

Demonstrating Python concepts tested: dictionary iteration and custom sorting lambda functions.

```python
# Ranking in descending order of utility using a lambda function:
ranked = sorted(evaluated_paths, key=lambda ev: ev.utility, reverse=True)

# Direct selection of the optimal path:
best_path = max(evaluated_paths, key=lambda ev: ev.utility)
```

In [ ]:
agent = UtilityBasedAgent()
evaluations = agent.evaluate_all_paths(DEFAULT_PREDEFINED_PATHS)

# Sort using lambda function
ranked = sorted(evaluations, key=lambda ev: ev.utility, reverse=True)

print("Paths Ranked by Utility (Descending):")
for rank, ev in enumerate(ranked, start=1):
    print(f"  Rank {rank}: {ev.path_str:<10} Utility: {ev.utility:>5.1f} (Cost: {ev.total_cost:.1f})")

# Selection via max() with lambda key
optimal = max(evaluations, key=lambda ev: ev.utility)
print(f"\nOptimal Choice via max(key=lambda): {optimal.path_str} with Utility {optimal.utility}")

## 4. Optional Challenge: Battery Capacity Constraint

In realistic robotic systems, the agent operates under finite energy resources.
If a path's cost exceeds the battery capacity limit (e.g., $5.0$ units), the path is rejected as infeasible regardless of its theoretical desirability.

In [ ]:
battery_limit = 5.0
challenge_result = run_utility_based_simulation(battery_limit=battery_limit)

print(f"Battery Threshold: {battery_limit} units\n")
print(f"{'Path':<12} | {'Cost':<6} | {'Battery Limit':<14} | {'Decision'}")
print("-" * 55)
for ev in challenge_result.evaluated_paths:
    decision = "ACCEPTED (Feasible)" if ev.is_feasible else "REJECTED (Battery Depleted)"
    print(f"{ev.path_str:<12} | {ev.total_cost:<6.1f} | {battery_limit:<14.1f} | {decision}")

print(f"\nSelected Feasible Path: {challenge_result.selected_path.path_str}")
print(f"Utility Score:          {challenge_result.selected_path.utility}")

## 5. Automated Graph Path Discovery (DFS)

Using Depth-First Search with backtracking to auto-discover all acyclic candidate paths between Start `A` and Goal `D`:

In [ ]:
discovered_paths = find_all_paths(DEFAULT_GRAPH, "A", "D")
print(f"Discovered {len(discovered_paths)} Directed Paths from 'A' to 'D':")
for p in discovered_paths:
    cost, details = calculate_path_cost(DEFAULT_GRAPH, p)
    utility = calculate_utility(cost)
    print(f"  {'-'.join(p):<12} -> Cost: {cost:<5.1f} | Utility: {utility:>5.1f}")

## 6. Viva Voce Reference & Study Guide

### 1. Utility vs. Goal-Based Agents
- **Goal-Based Agent**:
  - Evaluates states using a binary predicate: $G(s) \in \{\text{True}, \text{False}\}$.
  - Cannot distinguish between different ways of achieving the goal (e.g., whether a path costs \$4 or \$8, both achieve the goal).
  - Cannot balance conflicting objectives (e.g. speed vs. fuel consumption vs. passenger safety).
- **Utility-Based Agent**:
  - Maps states/trajectories to a continuous real-valued metric $U(s) \in \mathbb{R}$.
  - Quantifies *how desirable* a trajectory is ("degree of happiness").
  - Selects the optimal path when multiple valid solutions exist.

### 2. Expected Utility Theory & Maximum Expected Utility (MEU)
- **Deterministic Decision Rule**:
  $$a^* = \arg\max_{a} U(\text{Result}(a, s))$$
- **Stochastic Environments (Uncertainty)**:
  When action $a$ can lead to multiple possible outcome states $s'$ with probability $P(s' \mid a, e)$:
  $$\text{EU}(a \mid e) = \sum_{s'} P(s' \mid a, e) \cdot U(s')$$
- **Principle of Maximum Expected Utility (MEU)**:
  A rational agent should select the action that maximizes its expected utility:
  $$a^* = \arg\max_{a} \text{EU}(a \mid e)$$

### 3. Environment Performance Measures (PEAS Framework)
- Must be formulated **objectively** by the system designer from an external observer's standpoint.
- Evaluates actual environmental consequence (e.g., total energy consumed, survival rate, time to destination), not internal agent feelings.

### 4. Complexity Analysis
- **Time Complexity**: $O(P \times L)$
  - $P$: Number of candidate paths evaluated.
  - $L$: Average path length (number of nodes).
  - Edge lookups in Python dictionaries run in $O(1)$ amortized time. For a path of length $L$, cost summation requires $L - 1$ lookups.
  - Evaluating $P$ candidate paths requires $P \times O(L) = O(P \times L)$ operations.
- **Space Complexity**: $O(1)$ auxiliary space beyond storing the path evaluation records.